# Cross-Encoder Reranker — 2단계 검색 품질 향상

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Bi-Encoder**와 **Cross-Encoder**의 차이 및 각각의 역할 이해하기
- HuggingFace의 `BAAI/bge-reranker-v2-m3` 모델로 재순위화(Reranking) 구현하기
- `ContextualCompressionRetriever`를 Reranker 파이프라인에 통합하기

## 사전 지식

- VectorStoreRetriever와 임베딩 기반 검색의 작동 방식
- Bi-Encoder: 쿼리와 문서를 각각 독립적으로 인코딩하는 방식 (빠르지만 상호작용 없음)

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> BE[Bi-Encoder<br/>벡터 검색<br/>1차: k=10]:::process
    BE --> RE[Cross-Encoder<br/>Reranker<br/>2차: top_n=3]:::process
    RE --> R[정밀 선별된<br/>상위 3개 문서]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## Two-Stage Retrieval

| 단계 | 방식 | 역할 | 속도 |
|------|------|------|------|
| **1단계** | Bi-Encoder (벡터 검색) | 후보 문서 빠르게 추출 (k=10~50) | 빠름 |
| **2단계** | Cross-Encoder (Reranker) | 쿼리-문서 상호작용 분석, 정밀 재정렬 | 느리지만 정확 |

Cross-Encoder는 쿼리와 문서를 **쌍(pair)**으로 함께 분석해서 더 정확한 관련성을 계산해요.

> **주의**: Cross-Encoder 모델(특히 BAAI/bge-reranker-v2-m3)은 처음 실행 시 HuggingFace에서 모델을 다운로드하므로 시간이 걸릴 수 있어요.

> 🔑 **핵심 개념**: Bi-Encoder는 쿼리와 문서를 따로 인코딩해 코사인 유사도로 비교합니다(빠름). Cross-Encoder는 쿼리+문서를 함께 입력으로 받아 관련성 점수를 직접 계산합니다(정확). 속도와 정확도의 트레이드오프가 핵심입니다.

> 🎯 **강의 포인트**: Two-Stage Retrieval은 현업 검색 시스템의 표준입니다. 1단계에서 수십~수백 개 후보를 빠르게 추출하고, 2단계에서 소수를 정밀하게 재정렬합니다. Reranker를 모든 문서에 적용하면 너무 느려지므로 반드시 2단계로 나눠야 해요.

## 1. 환경 설정

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 필요한 라이브러리 import
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 환경 변수 로드
load_dotenv()

True

In [2]:
# 문서 출력 도우미 함수
def pretty_print_docs(docs, show_metadata=False):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(f"내용: {doc.page_content}")
        
        if show_metadata and doc.metadata:
            # relevance_score가 있으면 표시 (Reranker 결과인 경우)
            if 'relevance_score' in doc.metadata:
                print(f"관련성 점수: {doc.metadata['relevance_score']:.4f}")
            print(f"메타데이터: {doc.metadata}")
        
        print(f"{'─' * 100}\n")


## 2. 문서 준비

### 2.1 문서 로드 및 분할

In [3]:
# 1단계: 문서 로드
documents = TextLoader("./data/appendix-keywords.txt").load()

print(f"로드된 문서 수: {len(documents)}")
print(f"문서 길이: {len(documents[0].page_content)} 문자")


로드된 문서 수: 1
문서 길이: 5733 문자


In [4]:
# 2단계: 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 각 청크는 최대 500자
    chunk_overlap=100  # 인접 청크 간 100자 중복
)

texts = text_splitter.split_documents(documents)

print(f"분할된 청크 수: {len(texts)}")
print(f"\n첫 번째 청크 미리보기:")
print(f"{texts[0].page_content[:200]}...")


분할된 청크 수: 15

첫 번째 청크 미리보기:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정...


### 2.2 벡터스토어 생성 및 기본 검색

In [5]:
# ---------------------------------------------------
# HuggingFace 임베딩으로 FAISS 벡터스토어 생성 — Reranker 비교 기준선 확립
# ---------------------------------------------------

# ============================================================
# TODO: HuggingFaceEmbeddings로 임베딩하고 FAISS 벡터스토어를 만드세요
# 힌트: k=10으로 설정해야 Reranker가 충분한 후보 문서를 받을 수 있음
# 예상 결과: 벡터스토어 생성 완료 메시지 출력
# ============================================================

# 임베딩 모델 설정 (한국어/영어 모두 지원하는 모델)
# paraphrase-multilingual-mpnet-base-v2: 다국어 지원 Sentence-BERT 계열
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

# FAISS 벡터스토어 생성 및 검색기 설정
vectorstore = FAISS.from_documents(texts, embeddings_model)
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}  # Reranker가 재정렬할 후보 문서 수
)

print("✅ 벡터스토어 생성 완료")

✅ 벡터스토어 생성 완료


In [6]:
# ---------------------------------------------------
# Reranker 적용 전 기본 벡터 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 결과를 출력하세요
# 힌트: invoke(query) 호출 후 pretty_print_docs()로 출력
# 예상 결과: 10개 문서가 유사도 순으로 출력됨
# ============================================================

# 기본 검색 테스트
query = "자연어 처리에서 단어를 벡터로 변환하는 기술은 무엇인가요?"

print(f"🔍 검색 쿼리: {query}\n")

# Reranker 없이 기본 벡터 검색 수행
docs_before_rerank = base_retriever.invoke(query)

print("\n📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:")
pretty_print_docs(docs_before_rerank)

🔍 검색 쿼리: 자연어 처리에서 단어를 벡터로 변환하는 기술은 무엇인가요?


📊 [Reranker 적용 전] 벡터 유사도 기반 검색 결과:

총 10개 문서 검색됨

[문서 1]
내용: Crawling

정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec

정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
LLM (Large Language Model)
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
내용: Token

정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.
예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.
연관키워드: 토큰화, 자연어 처리, 구문 분석

Tokenizer

정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.
예시: "I love programming."이라는 문장을 ["I", "love", "programming", "."]으로 분할합니다.
연관키워드: 토큰화, 자연어 처리, 구문 분석

VectorStore

정의: 벡터스토어는 벡터 형식으로 변환된 데이터를 저장하는 시스템입니다. 이는 검색

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 3. Cross-Encoder Reranker 적용

이제 Cross-Encoder 모델을 사용하여 검색 결과를 재정렬합니다.

### 3.1 Cross-Encoder 모델 선택

다양한 Cross-Encoder 모델이 있습니다:

- `BAAI/bge-reranker-v2-m3`: 다국어 지원, 높은 정확도 (권장)
- `cross-encoder/ms-marco-MiniLM-L-6-v2`: 영어 특화, 빠른 속도
- `cross-encoder/ms-marco-MiniLM-L-12-v2`: 영어 특화, 높은 정확도

In [7]:
# 1단계: Cross-Encoder 모델 초기화
# BAAI/bge-reranker-v2-m3: 한국어/영어 등 다국어를 지원하는 고성능 Reranker
cross_encoder_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3"
)

# 2단계: Reranker 설정 (상위 3개만 선택)
# top_n=3: 10개 후보 중 가장 관련성 높은 3개만 최종 반환
# 🎯 강의 포인트: top_n은 base_retriever의 k보다 반드시 작아야 함
reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=3
)

# 3단계: 압축 검색기 생성
# base_retriever → 10개 추출 → reranker → 상위 3개 선별
# ⚠️ 주의: ContextualCompressionRetriever를 재사용해 Reranker를 압축기로 등록
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

print("✅ Cross-Encoder Reranker 설정 완료")
print(f"   - 모델: BAAI/bge-reranker-v2-m3")
print(f"   - 초기 검색: 10개 문서")
print(f"   - 최종 반환: 상위 3개 문서")

✅ Cross-Encoder Reranker 설정 완료
   - 모델: BAAI/bge-reranker-v2-m3
   - 초기 검색: 10개 문서
   - 최종 반환: 상위 3개 문서


In [8]:
# ---------------------------------------------------
# Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 같은 쿼리를 검색하고 결과를 비교하세요
# 힌트: show_metadata=True로 관련성 점수(relevance_score)를 함께 출력
# 예상 결과: 10개 → 3개로 줄어들고 관련성 점수가 표시됨
# ============================================================

# Reranker 적용 검색 테스트
print(f"🔍 검색 쿼리: {query}\n")

# Reranker 적용 검색
docs_after_rerank = compression_retriever.invoke(query)

print("\n🎯 [Reranker 적용 후] Cross-Encoder 기반 재정렬 결과:")
pretty_print_docs(docs_after_rerank, show_metadata=True)

🔍 검색 쿼리: 자연어 처리에서 단어를 벡터로 변환하는 기술은 무엇인가요?


🎯 [Reranker 적용 후] Cross-Encoder 기반 재정렬 결과:

총 3개 문서 검색됨

[문서 1]
내용: Crawling

정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec

정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
LLM (Large Language Model)
메타데이터: {'source': './data/appendix-keywords.txt'}
────────────────────────────────────────────────────────────────────────────────────────────────────

[문서 2]
내용: Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합

## 4. 결과 비교 및 분석

Reranker 적용 전후를 비교하여 효과를 확인합니다.

In [9]:
# ---------------------------------------------------
# Reranker 적용 전후 결과 비교 분석
# ---------------------------------------------------
print("\n" + "=" * 100)
print("📊 Reranker 효과 비교")
print("=" * 100)

print(f"\n[검색 쿼리]")
print(f"  {query}")

print(f"\n[Reranker 적용 전] - 상위 3개")
for i, doc in enumerate(docs_before_rerank[:3], 1):
    # 첫 100자만 표시
    preview = doc.page_content.replace('\n', ' ')[:100]
    print(f"  {i}. {preview}...")

print(f"\n[Reranker 적용 후] - 상위 3개 (관련성 점수 포함)")
for i, doc in enumerate(docs_after_rerank, 1):
    preview = doc.page_content.replace('\n', ' ')[:100]
    score = doc.metadata.get('relevance_score', 'N/A')
    if isinstance(score, float):
        print(f"  {i}. [점수: {score:.4f}] {preview}...")
    else:
        print(f"  {i}. {preview}...")

print("\n💡 분석:")
print("  - Cross-Encoder는 쿼리와 문서를 함께 분석하여 더 정확한 관련성 평가")
print("  - 관련성 점수가 높은 문서가 상위로 재정렬됨")
print("  - 최종 LLM에 전달되는 문서 품질 향상")


📊 Reranker 효과 비교

[검색 쿼리]
  자연어 처리에서 단어를 벡터로 변환하는 기술은 무엇인가요?

[Reranker 적용 전] - 상위 3개
  1. Crawling  정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다. 예시: 구글 검색 ...
  2. Token  정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다. 예시: 문장 "나는 학교에 간다"를 "나는"...
  3. Parser  정의: 파서는 주어진 데이터(문자열, 파일 등)를 분석하여 구조화된 형태로 변환하는 도구입니다. 이는 프로그래밍 언어의 구문 분석이나 파일 데이터 처리에 사용됩니다....

[Reranker 적용 후] - 상위 3개 (관련성 점수 포함)
  1. Crawling  정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다. 예시: 구글 검색 ...
  2. Semantic Search  정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다. 예시: 사용자가 "태...
  3. Token  정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다. 예시: 문장 "나는 학교에 간다"를 "나는"...

💡 분석:
  - Cross-Encoder는 쿼리와 문서를 함께 분석하여 더 정확한 관련성 평가
  - 관련성 점수가 높은 문서가 상위로 재정렬됨
  - 최종 LLM에 전달되는 문서 품질 향상


## 5. 다양한 쿼리로 테스트

여러 유형의 쿼리로 Reranker의 효과를 확인합니다.

In [10]:
# ---------------------------------------------------
# 다양한 쿼리로 Reranker 성능 테스트
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 각각 compression_retriever로 검색하세요
# 힌트: for 루프로 test_queries를 순회하며 invoke() 호출
# 예상 결과: 각 쿼리별 상위 3개 문서와 관련성 점수 출력
# ============================================================

# 테스트 쿼리 목록
test_queries = [
    "머신러닝에서 대규모 언어 모델이란?",
    "데이터를 저장하고 빠르게 검색하는 시스템은?",
    "웹에서 자동으로 정보를 수집하는 기술을 알려줘"
]

print("\n" + "=" * 100)
print("🧪 다양한 쿼리 테스트")
print("=" * 100)

for idx, test_query in enumerate(test_queries, 1):
    print(f"\n\n{'─' * 100}")
    print(f"[테스트 {idx}] 쿼리: {test_query}")
    print(f"{'─' * 100}")
    
    # Reranker 적용
    reranked_docs = compression_retriever.invoke(test_query)
    
    print(f"\n✅ 최종 선택된 문서 (상위 {len(reranked_docs)}개):")
    
    for i, doc in enumerate(reranked_docs, 1):
        # 첫 80자만 표시
        preview = doc.page_content.replace('\n', ' ')[:80]
        score = doc.metadata.get('relevance_score', 0)
        print(f"\n  [{i}] 점수: {score:.4f}")
        print(f"      내용: {preview}...")


🧪 다양한 쿼리 테스트


────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 1] 쿼리: 머신러닝에서 대규모 언어 모델이란?
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ 최종 선택된 문서 (상위 3개):

  [1] 점수: 0.0000
      내용: 정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다...

  [2] 점수: 0.0000
      내용: Crawling  정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 ...

  [3] 점수: 0.0000
      내용: 판다스 (Pandas)  정의: 판다스는 파이썬 프로그래밍 언어를 위한 데이터 분석 및 조작 도구를 제공하는 라이브러리입니다. 이는 데이터 분석...


────────────────────────────────────────────────────────────────────────────────────────────────────
[테스트 2] 쿼리: 데이터를 저장하고 빠르게 검색하는 시스템은?
────────────────────────────────────────────────────────────────────────────────────────────────────

✅ 최종 선택된 문서 (상위 3개):

  [1] 점수: 0.0000
      내용: SQL  정의: SQL(Structured Query Language)은 데이터베이스에서 데이터를 관리하기 위한 프로그래밍 언어입니다. 데이터 ...

  [2] 점수: 0.0000

## 6. 핵심 정리

### Reranker의 주요 장점

1. **정확도 향상**
   - Cross-Encoder가 쿼리-문서 간 상호작용을 직접 분석
   - Bi-Encoder보다 훨씬 정확한 관련성 평가

2. **비용 효율성**
   - 초기 검색: 많은 후보를 빠르게 추출
   - 재정렬: 소수 문서만 정밀 분석
   - LLM 입력 토큰 수 절감

3. **유연한 통합**
   - 기존 검색 시스템에 쉽게 추가 가능
   - 다양한 Cross-Encoder 모델 선택 가능

### 사용 시 고려사항

- **초기 검색 수 (k)**: 충분한 후보 확보 (보통 10~50개)
- **최종 문서 수 (top_n)**: LLM 컨텍스트 크기 고려 (보통 3~5개)
- **모델 선택**: 
  - 다국어: `BAAI/bge-reranker-v2-m3`
  - 영어 특화: `cross-encoder/ms-marco-MiniLM-L-12-v2`
  - 속도 중시: 경량 모델 (L-6 버전)

### Two-Stage Retrieval 권장 설정

```python
# Stage 1: 벡터 검색 (빠르게 후보 추출)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Stage 2: Cross-Encoder 재정렬 (정확하게 선별)
reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=5)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)
```

> 💡 **실무 팁**: Cross-Encoder는 GPU가 있으면 훨씬 빠릅니다. CPU 환경에서 느리다면 더 경량 모델(L-6 버전)을 선택하거나, 클라우드 기반 Reranker(Cohere, Jina)로 전환하세요.

> ⚠️ **자주 하는 실수**: `k`를 너무 작게 설정하면 Reranker가 재정렬할 후보가 부족해집니다. `k`는 `top_n`의 3~5배 이상으로 설정하는 것을 권장합니다.

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

### Two-Stage Retrieval 권장 설정

```python
# 1단계: 빠른 후보 추출
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# 2단계: 정밀 재정렬 (상위 5개만 최종 선택)
reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=5)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)
```

### 다음 노트북 예고

**02-Cohere-Reranker**: 로컬 모델 대신 Cohere의 클라우드 API를 활용한 Reranker를 배워요. 인프라 관리 없이 프로덕션 수준의 재순위화를 구현합니다.